## Step 1 — Access and inspect Daymet

The fifth project data source is **Daymet V4**, a gridded daily surface-weather dataset produced by NASA/ORNL. Unlike the static terrain information from 3DEP or the periodic vegetation snapshot from LANDFIRE, Daymet contains a new raster observation for each day and therefore introduces a substantial temporal dimension.

For the wildfire-recovery model, Daymet will provide broader-scale climate context describing both:

1. the climate a location typically experienced before the focal wildfire, and
2. the actual weather conditions experienced through the first complete post-fire growing season.

The primary model must remain leakage-safe: climate information occurring after the end of Season 1 cannot be used as a predictor of Year-5 NBR.

Before defining climate features or aggregation windows, this step verifies the Earth Engine source and establishes its basic structure:

- collection availability and temporal cadence,
- available climate variables and their units,
- spatial resolution and native projection,
- coverage of the Highline fire,
- and the structure of individual daily images.

No final Daymet feature definitions are locked during this step.

### Step 1A — Load the Daymet collection and inspect its basic structure

First, load the Daymet V4 daily ImageCollection and inspect it at a high level before performing any climate aggregation.

In [1]:
import ee

ee.Initialize(project="ee-wildfire-recovery-vasant")


# Highline fire geometry
mtbs = ee.FeatureCollection(
    "USFS/GTAC/MTBS/burned_area_boundaries/v1"
)

highline = mtbs.filter(
    ee.Filter.eq("Event_ID", "ID4549011522720170729")
).first()

highline_geom = highline.geometry()


# Daymet V4
daymet = ee.ImageCollection(
    "NASA/ORNL/DAYMET_V4"
)

daymet_highline = daymet.filterBounds(highline_geom)


# Basic collection inspection
first_image = ee.Image(daymet_highline.first())

first_date = ee.Date(
    daymet_highline.aggregate_min("system:time_start")
).format("YYYY-MM-dd")

last_date = ee.Date(
    daymet_highline.aggregate_max("system:time_start")
).format("YYYY-MM-dd")


print("Highline fire:", highline.get("Incid_Name").getInfo())
print("Daymet images intersecting Highline:", daymet_highline.size().getInfo())
print("First available date:", first_date.getInfo())
print("Last available date:", last_date.getInfo())
print("Bands:", first_image.bandNames().getInfo())

Highline fire: HIGHLINE
Daymet images intersecting Highline: 16790
First available date: 1980-01-01
Last available date: 2025-12-31
Bands: ['dayl', 'prcp', 'srad', 'swe', 'tmax', 'tmin', 'vp']


Daymet V4 loaded successfully for the Highline study area.

The collection contains **16,790 daily images spanning January 1, 1980 through December 31, 2025**, confirming the expected one-image-per-day temporal structure.

Each daily image contains seven climate bands:

- `dayl`
- `prcp`
- `srad`
- `swe`
- `tmax`
- `tmin`
- `vp`

Because the Daymet images cover the Highline region throughout the full available record, the spatial `filterBounds` operation retains the complete daily time series rather than selecting only a small subset of dates.

This confirms that Daymet can provide both multi-year pre-fire climate information and daily weather observations through the first complete post-fire growing season.

### Step 1B — Inspect Daymet variables, units, and representative daily values

Daymet stores seven climate variables in every daily image. Before deciding how any variable should be aggregated into model features, inspect their physical meaning, units, and representative values over the Highline fire.

A single date within Highline's first complete post-fire growing season is used as an illustrative example: **July 15, 2018**.

For each band, calculate the spatial mean, minimum, and maximum across the Highline perimeter for intuition.

The Daymet variables are:

- `dayl` — daylight duration, in seconds
- `prcp` — total daily precipitation, in millimeters
- `srad` — incident shortwave radiation averaged over the daylight period, in W/m²
- `swe` — snow-water equivalent, in kg/m²
- `tmax` — daily maximum 2-meter air temperature, in °C
- `tmin` — daily minimum 2-meter air temperature, in °C
- `vp` — daily average partial pressure of water vapor, in Pa

No variable is selected or excluded for the final model at this stage.

In [2]:
# Representative Daymet day within Highline Season 1
sample_date = "2018-07-15"

sample_day = ee.Image(
    daymet_highline
    .filterDate(sample_date, "2018-07-16")
    .first()
)


# Mean, minimum, and maximum across the Highline perimeter
daily_reducer = (
    ee.Reducer.mean()
    .combine(
        reducer2=ee.Reducer.minMax(),
        sharedInputs=True
    )
)

sample_stats = sample_day.reduceRegion(
    reducer=daily_reducer,
    geometry=highline_geom,
    scale=1000,
    maxPixels=1e7
).getInfo()


# Units from the official Daymet documentation
daymet_units = {
    "dayl": "seconds",
    "prcp": "mm",
    "srad": "W/m²",
    "swe": "kg/m²",
    "tmax": "°C",
    "tmin": "°C",
    "vp": "Pa",
}


print("Sample date:", sample_date)
print()

for band, unit in daymet_units.items():
    print(
        f"{band:5s} | "
        f"mean = {sample_stats[f'{band}_mean']:.2f} {unit:7s} | "
        f"min = {sample_stats[f'{band}_min']:.2f} | "
        f"max = {sample_stats[f'{band}_max']:.2f}"
    )

Sample date: 2018-07-15

dayl  | mean = 54491.88 seconds | min = 54440.88 | max = 54560.72
prcp  | mean = 0.00 mm      | min = 0.00 | max = 0.00
srad  | mean = 538.53 W/m²    | min = 483.44 | max = 691.09
swe   | mean = 23.15 kg/m²   | min = 0.00 | max = 433.82
tmax  | mean = 27.24 °C      | min = 22.31 | max = 31.36
tmin  | mean = 8.53 °C      | min = 5.14 | max = 9.93
vp    | mean = 965.84 Pa      | min = 742.39 | max = 1155.56


#### Step 1B observations

A representative Daymet image from **July 15, 2018**, within Highline's first complete post-fire growing season, produced physically plausible climate values across the fire perimeter.

Mean daylight duration was approximately **54,492 seconds (~15.1 hours)**, with little spatial variation across the fire. No precipitation was recorded anywhere within Highline on the selected date.

Daily maximum temperatures averaged **27.2°C** across the fire, ranging from approximately **22.3°C to 31.4°C**, while daily minimum temperatures averaged **8.5°C**, ranging from approximately **5.1°C to 9.9°C**. These statistics represent spatial variation in each grid cell's daily maximum or minimum temperature rather than the temporal temperature range of the fire as a whole.

Mean daylight-period shortwave radiation was approximately **539 W/m²**, and mean vapor pressure was approximately **966 Pa**, with both variables showing spatial variation across the fire.

Snow-water equivalent (`swe`) averaged approximately **23 kg/m²** but ranged from zero to approximately **434 kg/m²**. Because 1 kg/m² of water corresponds approximately to 1 mm of water depth, the higher values indicate substantial remaining modeled snowpack in some locations rather than snowfall occurring on the selected day. The large spatial range should be examined further during temporal and spatial validation.

Overall, the representative daily values are physically plausible and confirm that Daymet provides meaningful spatial variation across Highline even at its substantially coarser resolution relative to Landsat, LANDFIRE, and 3DEP.

These observations remain exploratory; no Daymet variables have yet been selected as final model features.

### Step 1C — Inspect Daymet's spatial grid, projection, and Highline coverage

Daymet has a substantially coarser native spatial resolution than the other raster sources used in the project. Before eventually associating climate information with the common 90 m modeling grid, we inspect Daymet's native projection and nominal pixel scale directly in Earth Engine.

This step also verifies that a representative Daymet climate band provides complete spatial coverage of the Highline fire perimeter.

The objectives are to determine:

- Daymet's native coordinate reference system (CRS),
- its native raster transform,
- its nominal spatial scale,
- and whether valid Daymet climate values cover the full Highline study area.

No resampling to the future 90 m modeling grid is performed here. Daymet will remain a coarse-scale climate source even when its values are later associated with 90 m model patches.

In [3]:
# Inspect the native Daymet grid using tmax as a representative band
daymet_projection = sample_day.select("tmax").projection()

projection_info = daymet_projection.getInfo()
nominal_scale = daymet_projection.nominalScale().getInfo()

print("Projection type: Lambert Conformal Conic")
print("CRS authority code:", projection_info.get("crs", "Not provided; projection stored as WKT"))
print("Transform:", projection_info["transform"])
print(f"Nominal scale: {nominal_scale:.2f} m")


# Check valid Daymet coverage across Highline
highline_area_m2 = highline_geom.area(maxError=1).getInfo()

valid_daymet_area_m2 = (
    ee.Image.pixelArea()
    .updateMask(sample_day.select("tmax").mask())
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=highline_geom,
        scale=1000,
        maxPixels=1e7
    )
    .get("area")
    .getInfo()
)

coverage_pct = 100 * valid_daymet_area_m2 / highline_area_m2

print()
print(f"Highline vector area: {highline_area_m2 / 1e6:.2f} km²")
print(f"Valid Daymet area:   {valid_daymet_area_m2 / 1e6:.2f} km²")
print(f"Daymet coverage:     {coverage_pct:.2f}%")

Projection type: Lambert Conformal Conic
CRS authority code: Not provided; projection stored as WKT
Transform: [1000, 0, -5802750, 0, -1000, 4984500]
Nominal scale: 1000.00 m

Highline vector area: 378.31 km²
Valid Daymet area:   379.03 km²
Daymet coverage:     100.19%


#### Step 1C observations

Daymet uses a Lambert Conformal Conic projected coordinate system with coordinates measured in meters. Earth Engine provides the full coordinate-system definition as WKT rather than a short EPSG identifier. (Daymet uses its own meter-based map grid, with 1 km cells.)

The native raster transform is:

`[1000, 0, -5802750, 0, -1000, 4984500]`

and Earth Engine reports a nominal scale of **1,000 m**. Unlike the geographic-degree grid encountered with 3DEP, Daymet therefore uses a projected meter-based grid with approximately 1 km × 1 km native cells.

This resolution is substantially coarser than the future 90 m modeling grid. Consequently, multiple nearby 90 m wildfire-recovery patches will often share the same or very similar Daymet climate information. Daymet should therefore be interpreted as broader-scale climate context rather than fine-scale 90 m weather measurement.

The Highline polygon has an area of approximately **378.31 km²**. A representative Daymet `tmax` image produced approximately **379.03 km²** of valid raster area when evaluated at 1 km scale. The resulting nominal coverage calculation of 100.19% reflects minor rasterization and boundary effects from evaluating a coarse 1 km grid against an irregular vector perimeter rather than literal coverage greater than 100%.

The practical conclusion is that Daymet provides **effectively complete spatial coverage of Highline**, with no meaningful missing-data problem for the representative date.

No resampling to the future common 90 m modeling grid is performed during this POC.

### Step 2A — Understand the physical meaning of the Daymet variables


The seven Daymet variables are:

- `prcp` — daily precipitation
- `tmax` — daily maximum air temperature
- `tmin` — daily minimum air temperature
- `vp` — daily average water-vapor pressure
- `swe` — snow-water equivalent
- `srad` — daylight-period shortwave solar radiation
- `dayl` — daylight duration

Each variable is considered below in terms of what it measures, how it could plausibly relate to wildfire recovery, and whether it is likely to be a primary, secondary, or low-priority modeling input.

### Step 2B — Inspect seasonal climate behavior through time

A single Daymet image confirms that the variables are physically plausible on one date, but it does not show how those variables behave through the seasons.

To develop intuition about Daymet's temporal structure, summarize the daily climate record for **calendar year 2018**, which contains Highline's first complete post-fire growing season (June 1–September 30, 2018).

Monthly summaries are used for exploration:

- precipitation (`prcp`) is **summed across days** within each month because precipitation represents an accumulated water input;
- maximum and minimum temperature (`tmax`, `tmin`) are averaged across days;
- vapor pressure (`vp`) is averaged across days;
- snow-water equivalent (`swe`) is averaged across days because it represents the amount of water stored in the snowpack at a given time rather than daily snowfall;
- shortwave radiation (`srad`) is averaged across days;
- daylight duration (`dayl`) is averaged and converted from seconds to hours.

For each monthly climate surface, values are then spatially averaged across the Highline perimeter.

These summaries are exploratory.

In [4]:
import pandas as pd


# Build monthly Highline climate summaries for 2018
def summarize_month(month):
    start = ee.Date.fromYMD(2018, month, 1)
    end = start.advance(1, "month")

    monthly = daymet_highline.filterDate(start, end)

    # Temporal aggregation within each Daymet pixel
    summary_image = ee.Image.cat([
        monthly.select("prcp")
            .sum()
            .rename("prcp_total_mm"),

        monthly.select("tmax")
            .mean()
            .rename("tmax_mean_c"),

        monthly.select("tmin")
            .mean()
            .rename("tmin_mean_c"),

        monthly.select("vp")
            .mean()
            .rename("vp_mean_pa"),

        monthly.select("swe")
            .mean()
            .rename("swe_mean_kg_m2"),

        monthly.select("srad")
            .mean()
            .rename("srad_mean_w_m2"),

        monthly.select("dayl")
            .mean()
            .divide(3600)
            .rename("dayl_mean_hours"),
    ])

    # Spatial mean across Highline
    stats = summary_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=highline_geom,
        scale=1000,
        maxPixels=1e7
    )

    return ee.Feature(
        None,
        stats
    ).set({
        "month": month,
        "days": end.difference(start, "day")
    })


monthly_fc = ee.FeatureCollection(
    ee.List.sequence(1, 12).map(summarize_month)
)

monthly_info = monthly_fc.getInfo()


# Convert the returned Earth Engine features to a readable table
rows = [feature["properties"] for feature in monthly_info["features"]]

monthly_df = pd.DataFrame(rows).sort_values("month")

monthly_df["month"] = monthly_df["month"].astype(int)
monthly_df["days"] = monthly_df["days"].astype(int)

monthly_df = monthly_df[
    [
        "month",
        "days",
        "prcp_total_mm",
        "tmax_mean_c",
        "tmin_mean_c",
        "vp_mean_pa",
        "swe_mean_kg_m2",
        "srad_mean_w_m2",
        "dayl_mean_hours",
    ]
].round(2)

monthly_df

,month,days,prcp_total_mm,tmax_mean_c,tmin_mean_c,vp_mean_pa,swe_mean_kg_m2,srad_mean_w_m2,dayl_mean_hours
0,1,31,114.28,1.60,-7.68,355.28,118.05,188.24,8.97
1,2,28,132.92,-1.47,-11.35,294.93,174.19,277.63,10.14
2,3,31,100.82,3.81,-7.90,353.10,257.47,389.46,11.68
3,4,30,124.85,7.92,-3.41,487.94,271.93,483.70,13.31
4,5,31,106.15,15.21,2.62,745.38,202.16,487.69,14.70
5,6,30,81.34,17.21,3.83,811.90,96.42,479.52,15.41
6,7,31,0.00,25.66,7.63,973.82,24.60,516.53,15.07
7,8,31,35.22,23.13,6.17,868.79,1.72,401.39,13.85
8,9,30,13.79,18.09,1.29,468.59,0.29,376.98,12.27
9,10,31,86.13,9.65,-1.73,534.30,0.45,223.51,10.64


#### Step 2B observations

Monthly Daymet summaries for calendar year 2018 show a physically coherent seasonal climate cycle across Highline.

Precipitation is relatively high during the cool season and substantially lower during summer. July recorded essentially no precipitation across Highline, while August and September remained comparatively dry. The June–September period therefore appears substantially drier than much of the surrounding year.

Maximum and minimum temperatures follow the expected seasonal cycle, with strongly cold winter conditions and the warmest conditions during July and August. July's spatially averaged daily maximum temperature was approximately 25.7°C and its average daily minimum approximately 7.6°C.

Vapor pressure also increases substantially during the warm season, reaching its highest monthly mean in July. Because July simultaneously experienced essentially no precipitation, this demonstrates that raw vapor pressure should not be interpreted directly as a measure of environmental dryness. A derived atmospheric-demand metric such as vapor pressure deficit may therefore be worth investigating.

Snow-water equivalent (`swe`) shows a clear mountain snowpack cycle. Mean SWE rises through winter, reaches its highest values around March–April, and then declines rapidly during spring and summer, approaching zero by late summer. The July monthly mean (~24.6 kg/m²) closely agrees with the representative July 15 value observed earlier (~23.2 kg/m²), supporting the consistency of the daily and monthly summaries.

Shortwave radiation increases strongly from winter into spring and summer before declining again during fall. Daylight duration follows the expected astronomical cycle, increasing from approximately 9 hours in January to more than 15 hours in June before declining toward winter.

Overall, all seven Daymet variables exhibit plausible seasonal behavior. The results support continued investigation of precipitation, temperature, atmospheric moisture/dryness, snowpack, and radiation, while also suggesting that daylight duration may provide relatively little independent predictive information.

### Step 2C — Establish candidate climate concepts for feature engineering

The Daymet source contains seven daily variables, but the final model should not mechanically include every available band. Instead, retain climate concepts that are physically interpretable, potentially relevant to post-fire vegetation recovery, and not unnecessarily redundant with one another.

Based on the source definitions and the seasonal behavior observed for Highline, the candidate climate concepts are grouped into four categories:

1. **Core candidates** — likely to represent important recovery mechanisms.
2. **Derived candidates** — quantities that may be more ecologically meaningful than the raw source variable.
3. **Exploratory candidates** — scientifically plausible but potentially redundant or lower priority.
4. **Low-priority variables** — available in Daymet but unlikely to provide much independent predictive information.

#### Step 2C candidate-feature assessment

The Daymet variables were grouped according to their likely ecological relevance and potential redundancy.

**Precipitation** is retained as a core climate concept because it directly represents water input and is likely to influence post-fire vegetation establishment and growth.

**Temperature** is also retained as a core concept. Daily maximum and minimum temperature may support summaries of general thermal conditions, heat stress, cold conditions, or temperature anomalies, although the final representation remains undecided.

**Atmospheric dryness** is a strong derived-feature candidate. Raw Daymet vapor pressure represents atmospheric moisture content but does not directly measure plant atmospheric water demand. Because Daymet also provides temperature, a derived metric such as vapor pressure deficit (VPD) may provide a more interpretable measure of atmospheric dryness and vegetation water stress.

**Snow-water equivalent** is retained as an exploratory candidate because seasonal snowpack may influence spring and early-summer water availability. However, its additional predictive value beyond precipitation, temperature, and elevation must be evaluated empirically.

**Shortwave solar radiation** is also retained as exploratory. It has plausible ecological relevance through photosynthesis, surface heating, and evapotranspiration, but may overlap substantially with season, temperature, latitude, and terrain variables.

**Daylight duration** is considered low priority because it is largely determined by latitude and calendar date rather than short-term weather variability.

The current priority hierarchy is therefore:

- core: precipitation and temperature,
- strong derived candidate: atmospheric dryness / VPD,
- exploratory: SWE and solar radiation,
- likely unnecessary: day length.

### Step 3A — Establish leakage-safe temporal boundaries

Daymet provides daily climate observations across many decades, but not every available date can be used as a predictor.

For the Highline wildfire, the focal fire occurred in **2017**. The first complete post-fire growing season is therefore defined as **June 1 through September 30, 2018**, and the primary model's information cutoff occurs at the end of September 2018.

The temporal record can be separated conceptually into:

1. **Pre-fire historical climate** — climate observed before the focal wildfire, used to characterize the site's background climate.
2. **Focal-fire / post-fire antecedent period** — climate occurring after the fire but before or leading into the first complete growing season. This information is temporally available to the primary model, although its exact feature treatment remains undecided.
3. **Season 1 climate** — June 1 through September 30, 2018, representing weather during the first complete post-fire growing season.
4. **Future climate** — climate occurring after September 30, 2018. These observations are forbidden as predictors because they occur after the model's intended prediction cutoff.

The early ablation model is intentionally more restrictive and will not receive Season-1 climate information. Its climate predictors will be limited to leakage-safe historical information defined in the following subsections.

This step verifies the relevant dates and Daymet observation counts before selecting exact feature windows.

In [5]:
# Highline temporal boundaries
fire_year = 2017

season1_start = ee.Date("2018-06-01")
season1_end = ee.Date("2018-10-01")

prediction_cutoff = ee.Date("2018-10-01")


# Useful reference periods
last_prefire_growing_start = ee.Date("2016-06-01")
last_prefire_growing_end = ee.Date("2016-10-01")

season1_daymet = daymet_highline.filterDate(
    season1_start,
    season1_end
)

last_prefire_growing_daymet = daymet_highline.filterDate(
    last_prefire_growing_start,
    last_prefire_growing_end
)


print("Highline focal fire year:", fire_year)

print()
print(
    "Last complete pre-fire growing season:",
    last_prefire_growing_start.format("YYYY-MM-dd").getInfo(),
    "to",
    last_prefire_growing_end.advance(-1, "day").format("YYYY-MM-dd").getInfo()
)
print(
    "Daymet observations:",
    last_prefire_growing_daymet.size().getInfo()
)

print()
print(
    "Season 1:",
    season1_start.format("YYYY-MM-dd").getInfo(),
    "to",
    season1_end.advance(-1, "day").format("YYYY-MM-dd").getInfo()
)
print(
    "Daymet observations:",
    season1_daymet.size().getInfo()
)

print()
print(
    "Primary-model prediction cutoff:",
    prediction_cutoff.advance(-1, "day").format("YYYY-MM-dd").getInfo()
)
print("Climate after this date is forbidden as a predictor.")

Highline focal fire year: 2017

Last complete pre-fire growing season: 2016-06-01 to 2016-09-30
Daymet observations: 122

Season 1: 2018-06-01 to 2018-09-30
Daymet observations: 122

Primary-model prediction cutoff: 2018-09-30
Climate after this date is forbidden as a predictor.


#### Step 3A observations

The temporal boundaries for Highline were verified successfully.

The last complete pre-fire growing season is **June 1–September 30, 2016**, while the first complete post-fire growing season (Season 1) is **June 1–September 30, 2018**. Daymet contains exactly **122 daily observations** within each period, confirming complete daily temporal coverage.

For the primary model, climate information is permitted through **September 30, 2018**. Any Daymet observations beginning October 1, 2018 or later are considered future information and cannot be used as predictors of Year-5 NBR.

The 2016 growing season is used here only as a reference period. The historical climate baseline will use multiple pre-fire years rather than a single year and is defined in the following subsection.

### Step 3B — Define the pre-fire historical climate baseline

The model needs a historical climate reference describing the typical environmental conditions experienced by each location before the focal wildfire.

A useful baseline should balance two competing goals:

- include enough years to avoid treating unusual weather in one or two years as the site's normal climate;
- remain recent enough to represent the climate regime experienced by the vegetation immediately preceding the focal fire.

Candidate rolling windows of **5, 10, and 20 complete pre-fire years** are therefore compared for Highline. Each candidate uses the same June 1–September 30 calendar window as Season 1 so that later climate anomalies compare equivalent parts of the annual cycle.

For the 2017 Highline fire, the candidate windows are:

- 5-year baseline: 2012–2016
- 10-year baseline: 2007–2016
- 20-year baseline: 1997–2016

The current working preference is a 10-year pre-fire baseline, but the comparison below checks whether that choice produces materially different climate summaries from shorter or longer windows.

This baseline is intended to characterize background climate rather than reconstruct vegetation condition; Landsat separately provides the recent pre-fire spectral history.

In [6]:
# Build one Jun-Sep climate summary for a given year
def growing_season_summary(year):
    start = ee.Date.fromYMD(year, 6, 1)
    end = ee.Date.fromYMD(year, 10, 1)

    season = daymet_highline.filterDate(start, end)

    summary_image = ee.Image.cat([
        season.select("prcp")
            .sum()
            .rename("prcp_total_mm"),

        season.select("tmax")
            .mean()
            .rename("tmax_mean_c"),

        season.select("tmin")
            .mean()
            .rename("tmin_mean_c"),

        season.select("vp")
            .mean()
            .rename("vp_mean_pa"),
    ])

    stats = summary_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=highline_geom,
        scale=1000,
        maxPixels=1e7
    )

    return ee.Feature(None, stats).set("year", year)


# Highline pre-fire growing seasons, 1997–2016
historical_fc = ee.FeatureCollection(
    ee.List.sequence(1997, 2016).map(growing_season_summary)
)

historical_info = historical_fc.getInfo()

historical_rows = [
    feature["properties"]
    for feature in historical_info["features"]
]

historical_df = (
    pd.DataFrame(historical_rows)
    .sort_values("year")
)

historical_df["year"] = historical_df["year"].astype(int)


# Compare 5-, 10-, and 20-year candidate baselines
baseline_rows = []

for n_years in [5, 10, 20]:
    subset = historical_df.tail(n_years)

    baseline_rows.append({
        "baseline_years": n_years,
        "start_year": int(subset["year"].min()),
        "end_year": int(subset["year"].max()),
        "prcp_mean_mm": subset["prcp_total_mm"].mean(),
        "tmax_mean_c": subset["tmax_mean_c"].mean(),
        "tmin_mean_c": subset["tmin_mean_c"].mean(),
        "vp_mean_pa": subset["vp_mean_pa"].mean(),
    })

baseline_df = pd.DataFrame(baseline_rows).round(2)

baseline_df

,baseline_years,start_year,end_year,prcp_mean_mm,tmax_mean_c,tmin_mean_c,vp_mean_pa
0,5,2012,2016,160.89,21.87,4.76,757.85
1,10,2007,2016,170.10,21.59,4.24,745.35
2,20,1997,2016,186.73,21.32,3.36,711.09


#### Step 3B observations

The 5-, 10-, and 20-year candidate historical baselines produced broadly similar but systematically different growing-season climate summaries for Highline.

Mean June–September precipitation was approximately:

- **160.9 mm** for 2012–2016,
- **170.1 mm** for 2007–2016,
- **186.7 mm** for 1997–2016.

The more recent baseline periods were also somewhat warmer. Mean daily maximum temperature increased from approximately **21.3°C** in the 20-year baseline to **21.9°C** in the 5-year baseline, while mean daily minimum temperature increased from approximately **3.4°C to 4.8°C**. Mean vapor pressure similarly increased in the more recent period.

These differences indicate that the choice of historical window is not completely neutral. Longer windows provide greater statistical stability but increasingly incorporate an older climate regime, while very short windows better represent recent conditions but are more sensitive to interannual variability.

For V1, a **rolling 10-complete-year pre-fire baseline** is adopted provisionally as a compromise between recency and stability. For Highline, this corresponds to **2007–2016**.

The baseline uses the same June 1–September 30 calendar window as Season 1 so that future climate anomalies compare equivalent portions of the annual cycle.

The 10-year choice should be audited across the full wildfire population during production dataset construction and EDA rather than treated as permanently validated from the Highline POC alone.

### Step 3C — Evaluate antecedent climate before Season 1

Season-1 climate during June–September directly describes weather experienced during the first complete post-fire growing season. However, vegetation entering that season may also be influenced by antecedent winter and spring conditions, including precipitation, temperature, and accumulated snowpack.

For Highline, two distinct post-fire climate periods are therefore examined:

1. **Antecedent period:** January 1–May 31, 2018
2. **Season 1:** June 1–September 30, 2018

The antecedent period is evaluated separately rather than automatically merging it with Season 1 so that winter/spring hydrologic conditions and growing-season weather remain interpretable.

Each 2018 period is compared with the corresponding calendar-month average from the rolling 10-year pre-fire baseline (2007–2016). This ensures that anomalies compare equivalent portions of the annual cycle.

This analysis is used to determine whether antecedent climate deserves its own candidate feature family in the primary model. It does not yet lock the exact final climate features.

The early ablation remains more restrictive and will use only pre-fire historical climate rather than post-fire antecedent or Season-1 climate.

In [7]:
# Summarize a custom climate period
def period_summary(start_date, end_date):
    period = daymet_highline.filterDate(start_date, end_date)

    summary_image = ee.Image.cat([
        period.select("prcp")
            .sum()
            .rename("prcp_total_mm"),

        period.select("tmax")
            .mean()
            .rename("tmax_mean_c"),

        period.select("tmin")
            .mean()
            .rename("tmin_mean_c"),

        period.select("vp")
            .mean()
            .rename("vp_mean_pa"),

        period.select("swe")
            .mean()
            .rename("swe_mean_kg_m2"),
    ])

    return summary_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=highline_geom,
        scale=1000,
        maxPixels=1e7
    ).getInfo()


# Actual 2018 periods
antecedent_2018 = period_summary(
    "2018-01-01",
    "2018-06-01"
)

season1_2018 = period_summary(
    "2018-06-01",
    "2018-10-01"
)


# Build matching 10-year historical baselines
def historical_period_baseline(start_month, start_day, end_month, end_day):
    rows = []

    for year in range(2007, 2017):
        start = f"{year}-{start_month:02d}-{start_day:02d}"

        # End date is exclusive.
        # Jan-May ends June 1 of the same year.
        # Jun-Sep ends October 1 of the same year.
        end = f"{year}-{end_month:02d}-{end_day:02d}"

        stats = period_summary(start, end)
        rows.append(stats)

    return pd.DataFrame(rows).mean(numeric_only=True)


antecedent_baseline = historical_period_baseline(
    1, 1,
    6, 1
)

season1_baseline = historical_period_baseline(
    6, 1,
    10, 1
)


# Compare actual 2018 climate with historical normal
comparison_rows = []

for period_name, actual, baseline in [
    ("Antecedent Jan-May", antecedent_2018, antecedent_baseline),
    ("Season 1 Jun-Sep", season1_2018, season1_baseline),
]:
    comparison_rows.append({
        "period": period_name,

        "prcp_actual_mm": actual["prcp_total_mm"],
        "prcp_baseline_mm": baseline["prcp_total_mm"],
        "prcp_anomaly_mm":
            actual["prcp_total_mm"] - baseline["prcp_total_mm"],

        "tmax_actual_c": actual["tmax_mean_c"],
        "tmax_baseline_c": baseline["tmax_mean_c"],
        "tmax_anomaly_c":
            actual["tmax_mean_c"] - baseline["tmax_mean_c"],

        "tmin_actual_c": actual["tmin_mean_c"],
        "tmin_baseline_c": baseline["tmin_mean_c"],
        "tmin_anomaly_c":
            actual["tmin_mean_c"] - baseline["tmin_mean_c"],

        "swe_actual": actual["swe_mean_kg_m2"],
        "swe_baseline": baseline["swe_mean_kg_m2"],
        "swe_anomaly":
            actual["swe_mean_kg_m2"] - baseline["swe_mean_kg_m2"],
    })


period_comparison_df = pd.DataFrame(comparison_rows).round(2)

period_comparison_df

,period,prcp_actual_mm,prcp_baseline_mm,prcp_anomaly_mm,tmax_actual_c,tmax_baseline_c,tmax_anomaly_c,tmin_actual_c,tmin_baseline_c,tmin_anomaly_c,swe_actual,swe_baseline,swe_anomaly
0,Antecedent Jan-May,579.02,479.25,99.77,5.54,5.85,-0.32,-5.44,-6.99,1.55,204.92,206.18,-1.26
1,Season 1 Jun-Sep,130.35,170.10,-39.75,21.08,21.59,-0.51,4.77,4.24,0.53,30.47,51.74,-21.28


#### Step 3C observations

Antecedent and Season-1 climate describe meaningfully different environmental conditions for Highline.

During **January–May 2018**, precipitation totaled approximately **579.0 mm**, compared with a 2007–2016 historical mean of approximately **479.3 mm**. The antecedent period was therefore roughly **99.8 mm (~20.8%) wetter than its recent historical normal**.

Antecedent temperatures were only modestly different from normal: mean daily maximum temperature was approximately 0.3°C cooler than the historical baseline, while mean daily minimum temperature was approximately 1.6°C warmer. Mean snow-water equivalent was nearly identical to the historical Jan–May baseline.

In contrast, Highline's **June–September 2018 Season 1** received approximately **130.4 mm** of precipitation compared with a historical growing-season mean of approximately **170.1 mm**, an anomaly of approximately **−39.8 mm (~23.4% below normal)**. Mean maximum and minimum temperatures differed from their historical baselines by less than 1°C.

This contrast demonstrates why antecedent winter/spring climate and first-growing-season climate should not automatically be merged into a single post-fire climate summary: Highline experienced an unusually wet antecedent period followed by a substantially drier-than-normal first growing season.

Antecedent climate is therefore retained as a candidate feature family for the primary model, alongside Season-1 climate and the historical climate baseline.

The exact antecedent window remains provisional. January–May is appropriate for the Highline POC because it occurs safely after the 2017 fire, but production windows should be audited relative to verified focal-fire timing rather than blindly assuming the same calendar boundary for every wildfire.

Likewise, the final representation of snowpack remains open. A late-spring or snowmelt-oriented SWE metric may ultimately be more informative than averaging SWE across the entire June–September growing season.

### Step 3D — Define the Daymet timing contract for the primary and early models

Daymet predictors must obey the information cutoff associated with each modeling setup.

For the **primary model**, prediction occurs after the end of the first complete post-fire growing season. Daymet may therefore describe:

1. **Historical climate** — a rolling 10-complete-year pre-fire baseline, using calendar periods matched to the climate feature being summarized.
2. **Antecedent post-fire climate** — climate occurring after the focal wildfire but before the beginning of Season 1. For the Highline POC, January 1–May 31, 2018 is used as a clean post-fire antecedent period.
3. **Season-1 climate** — June 1–September 30 of the first complete post-fire growing season.
4. **Climate anomalies** — comparisons between actual antecedent or Season-1 conditions and the corresponding historical calendar-period baseline.

No Daymet information occurring after the end of Season 1 may be used by the primary model.

For the **early ablation model**, Daymet is intentionally restricted to **pre-fire historical climate only**. It does not receive post-fire antecedent climate or Season-1 weather, preserving the intended comparison between predictions made without observing first-year recovery conditions and predictions made after Season 1.

Fire-period Daymet variables are not included in the core V1 feature contract. They are temporally legal and may be investigated later if verified fire timing permits a consistent definition and if they provide information beyond direct burn-impact measurements such as MTBS severity and Landsat-derived spectral change.

The antecedent-period definition remains provisional for production. Highline's January–May window is safely post-fire, but the full dataset must audit focal-fire timing before applying a common antecedent rule across all fires.

### Step 4A — Build core absolute climate features

The Daymet POC now transitions from source exploration to candidate feature construction.

The first feature group contains absolute climate summaries for three distinct temporal contexts:

1. **Historical climate baseline** — the rolling 10-complete-year pre-fire June–September average, representing the recent climate regime experienced before the focal wildfire.
2. **Antecedent climate** — January–May of the first post-fire year, representing winter and spring conditions leading into the first complete recovery growing season.
3. **Season-1 climate** — June–September of the first complete post-fire growing season.

For this initial core feature set, precipitation and temperature are retained because they represent two fundamental ecological controls: water availability and thermal conditions.

Precipitation is accumulated across days within each period, while daily maximum and minimum temperatures are averaged across days.

The resulting values are still summarized across the Highline perimeter for POC inspection. During production integration, equivalent climate surfaces will be associated geographically with individual 90 m modeling patches rather than assigning one Highline-wide value to every patch.

In [8]:
# Helper: summarize core climate variables for a given period
def core_period_summary(start_date, end_date):
    period = daymet_highline.filterDate(start_date, end_date)

    summary_image = ee.Image.cat([
        period.select("prcp")
            .sum()
            .rename("prcp_total_mm"),

        period.select("tmax")
            .mean()
            .rename("tmax_mean_c"),

        period.select("tmin")
            .mean()
            .rename("tmin_mean_c"),
    ])

    return summary_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=highline_geom,
        scale=1000,
        maxPixels=1e7
    ).getInfo()


# Historical Jun-Sep baseline: 2007-2016
historical_core_rows = []

for year in range(2007, 2017):
    stats = core_period_summary(
        f"{year}-06-01",
        f"{year}-10-01"
    )
    historical_core_rows.append(stats)

historical_core_df = pd.DataFrame(historical_core_rows)

historical_core = historical_core_df.mean(numeric_only=True)


# Actual post-fire periods
antecedent_core = core_period_summary(
    "2018-01-01",
    "2018-06-01"
)

season1_core = core_period_summary(
    "2018-06-01",
    "2018-10-01"
)


# Assemble one provisional Highline feature row
core_features = {
    "hist_prcp_mm": historical_core["prcp_total_mm"],
    "hist_tmax_c": historical_core["tmax_mean_c"],
    "hist_tmin_c": historical_core["tmin_mean_c"],

    "ante_prcp_mm": antecedent_core["prcp_total_mm"],
    "ante_tmax_c": antecedent_core["tmax_mean_c"],
    "ante_tmin_c": antecedent_core["tmin_mean_c"],

    "s1_prcp_mm": season1_core["prcp_total_mm"],
    "s1_tmax_c": season1_core["tmax_mean_c"],
    "s1_tmin_c": season1_core["tmin_mean_c"],
}

core_features_df = pd.DataFrame(
    [core_features],
    index=["Highline"]
).round(2)

core_features_df

,hist_prcp_mm,hist_tmax_c,hist_tmin_c,ante_prcp_mm,ante_tmax_c,ante_tmin_c,s1_prcp_mm,s1_tmax_c,s1_tmin_c
Highline,170.1,21.59,4.24,579.02,5.54,-5.44,130.35,21.08,4.77


#### Step 4A observations

The first core Daymet feature row was constructed successfully for Highline.

The 10-year historical June–September baseline contains approximately **170.1 mm** of precipitation, a mean daily maximum temperature of **21.6°C**, and a mean daily minimum temperature of **4.2°C**.

The January–May 2018 antecedent period received approximately **579.0 mm** of precipitation, with mean daily maximum and minimum temperatures of approximately **5.5°C** and **−5.4°C**, respectively.

During the June–September 2018 first complete post-fire growing season, Highline received approximately **130.4 mm** of precipitation, with mean daily maximum and minimum temperatures of approximately **21.1°C** and **4.8°C**.

These values form the first provisional absolute Daymet feature group. They represent historical climate context, antecedent post-fire conditions, and Season-1 weather separately rather than collapsing them into a single climate summary.

The Highline-wide values remain POC diagnostics; production features will be associated spatially with individual 90 m modeling patches.

### Step 4B — Derive climate anomalies and atmospheric dryness

Absolute climate values describe the environmental conditions experienced by a location, but they do not indicate whether those conditions were unusual relative to the site's recent climate.

For the antecedent and Season-1 periods, precipitation and temperature anomalies are therefore calculated relative to calendar-matched 10-year pre-fire baselines.

Atmospheric dryness is also represented using **vapor pressure deficit (VPD)** rather than treating raw Daymet vapor pressure as a direct dryness measure.

VPD approximates the difference between the amount of moisture the air could hold at saturation and the amount of water vapor actually present. Higher VPD generally corresponds to greater atmospheric evaporative demand and potentially greater vegetation water stress.

Daily saturation vapor pressure is estimated from Daymet minimum and maximum temperature, while Daymet `vp` supplies actual vapor pressure. Daily VPD is then averaged over each climate period.

These derived variables remain candidate features; their final predictive value will be evaluated during EDA and modeling.

In [9]:
import math


# Add daily vapor pressure deficit (VPD) to each Daymet image
def add_vpd(image):
    tmax = image.select("tmax")
    tmin = image.select("tmin")

    # Saturation vapor pressure (kPa) at Tmax and Tmin
    es_tmax = (
        tmax.multiply(17.27)
        .divide(tmax.add(237.3))
        .exp()
        .multiply(0.6108)
    )

    es_tmin = (
        tmin.multiply(17.27)
        .divide(tmin.add(237.3))
        .exp()
        .multiply(0.6108)
    )

    # Mean daily saturation vapor pressure
    es_mean = es_tmax.add(es_tmin).divide(2)

    # Daymet vapor pressure: Pa -> kPa
    ea = image.select("vp").divide(1000)

    # Vapor pressure deficit
    vpd = (
        es_mean.subtract(ea)
        .max(0)
        .rename("vpd_kpa")
    )

    return image.addBands(vpd)


daymet_with_vpd = daymet_highline.map(add_vpd)


# Mean VPD for a chosen period
def period_vpd_summary(start_date, end_date):
    period = daymet_with_vpd.filterDate(start_date, end_date)

    vpd_image = period.select("vpd_kpa").mean()

    return vpd_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=highline_geom,
        scale=1000,
        maxPixels=1e7
    ).get("vpd_kpa").getInfo()


# 10-year historical VPD baselines
hist_s1_vpd_values = []
hist_ante_vpd_values = []

for year in range(2007, 2017):
    hist_s1_vpd_values.append(
        period_vpd_summary(
            f"{year}-06-01",
            f"{year}-10-01"
        )
    )

    hist_ante_vpd_values.append(
        period_vpd_summary(
            f"{year}-01-01",
            f"{year}-06-01"
        )
    )

hist_s1_vpd = sum(hist_s1_vpd_values) / len(hist_s1_vpd_values)
hist_ante_vpd = sum(hist_ante_vpd_values) / len(hist_ante_vpd_values)


# Actual post-fire VPD
ante_vpd = period_vpd_summary(
    "2018-01-01",
    "2018-06-01"
)

s1_vpd = period_vpd_summary(
    "2018-06-01",
    "2018-10-01"
)


# Assemble anomaly + VPD candidate features
derived_features = {
    # Antecedent anomalies
    "ante_prcp_anom_mm":
        antecedent_core["prcp_total_mm"]
        - antecedent_baseline["prcp_total_mm"],

    "ante_tmax_anom_c":
        antecedent_core["tmax_mean_c"]
        - antecedent_baseline["tmax_mean_c"],

    "ante_tmin_anom_c":
        antecedent_core["tmin_mean_c"]
        - antecedent_baseline["tmin_mean_c"],

    # Season-1 anomalies
    "s1_prcp_anom_mm":
        season1_core["prcp_total_mm"]
        - season1_baseline["prcp_total_mm"],

    "s1_tmax_anom_c":
        season1_core["tmax_mean_c"]
        - season1_baseline["tmax_mean_c"],

    "s1_tmin_anom_c":
        season1_core["tmin_mean_c"]
        - season1_baseline["tmin_mean_c"],

    # VPD
    "hist_s1_vpd_kpa": hist_s1_vpd,

    "ante_vpd_kpa": ante_vpd,
    "ante_vpd_anom_kpa": ante_vpd - hist_ante_vpd,

    "s1_vpd_kpa": s1_vpd,
    "s1_vpd_anom_kpa": s1_vpd - hist_s1_vpd,
}


derived_features_df = pd.DataFrame(
    [derived_features],
    index=["Highline"]
).round(3)

derived_features_df

,ante_prcp_anom_mm,ante_tmax_anom_c,ante_tmin_anom_c,s1_prcp_anom_mm,s1_tmax_anom_c,s1_tmin_anom_c,hist_s1_vpd_kpa,ante_vpd_kpa,ante_vpd_anom_kpa,s1_vpd_kpa,s1_vpd_anom_kpa
Highline,99.772,-0.317,1.554,-39.75,-0.514,0.529,1.039,0.28,-0.036,0.977,-0.062


#### Step 4B observations

Climate-anomaly and vapor-pressure-deficit features were derived successfully.

The January–May 2018 antecedent period received approximately **99.8 mm more precipitation than its 10-year historical normal**. Mean maximum temperature was approximately **0.3°C cooler than normal**, while mean minimum temperature was approximately **1.6°C warmer**.

During Season 1, June–September 2018 precipitation was approximately **39.8 mm below the recent historical growing-season baseline**. Mean maximum and minimum temperatures differed from normal by less than 1°C.

Mean historical June–September VPD was approximately **1.04 kPa**, compared with approximately **0.98 kPa during Season 1**, producing a VPD anomaly of approximately **−0.06 kPa**. Thus, although Season 1 was substantially drier than normal in terms of precipitation, its average atmospheric evaporative demand was not unusually high. This demonstrates that precipitation and VPD represent distinct climate mechanisms.

Antecedent VPD was approximately **0.28 kPa**, slightly below its matching historical baseline.

Absolute climate values and climate anomalies are retained as candidate features for now. Because an anomaly is mathematically derived from the actual value and historical baseline, all three representations need not necessarily be retained in the final model. Redundancy and collinearity will be evaluated during EDA and modeling before the final feature set is locked.

### Step 4C — Evaluate exploratory climate variables and assemble the provisional feature set

Two Daymet variables remain exploratory rather than core: snow-water equivalent (`swe`) and shortwave solar radiation (`srad`).

SWE may provide additional hydrologic information by representing seasonal snowpack and stored water availability, especially in mountainous ecosystems. However, it may overlap with elevation, temperature, and precipitation.

Solar radiation has plausible ecological relevance through photosynthesis, surface heating, and evapotranspiration, but may also overlap with temperature, seasonality, latitude, and terrain exposure.

Rather than creating many derived variants, this POC evaluates a small number of interpretable summaries and then assembles the full provisional Daymet feature set.

The provisional set remains subject to spatial validation, redundancy checks, EDA, and held-out modeling before the final production features are locked.

In [10]:
# summarize exploratory Daymet variables for a period
def exploratory_period_summary(start_date, end_date):
    period = daymet_highline.filterDate(start_date, end_date)

    summary_image = ee.Image.cat([
        period.select("swe")
            .mean()
            .rename("swe_mean_kg_m2"),

        period.select("srad")
            .mean()
            .rename("srad_mean_w_m2"),
    ])

    return summary_image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=highline_geom,
        scale=1000,
        maxPixels=1e7
    ).getInfo()


# Historical Jun-Sep baseline
hist_exploratory_rows = []

for year in range(2007, 2017):
    stats = exploratory_period_summary(
        f"{year}-06-01",
        f"{year}-10-01"
    )
    hist_exploratory_rows.append(stats)

hist_exploratory_df = pd.DataFrame(hist_exploratory_rows)
hist_exploratory = hist_exploratory_df.mean(numeric_only=True)


# Actual post-fire periods
ante_exploratory = exploratory_period_summary(
    "2018-01-01",
    "2018-06-01"
)

s1_exploratory = exploratory_period_summary(
    "2018-06-01",
    "2018-10-01"
)


# Matching antecedent historical baseline
hist_ante_exploratory_rows = []

for year in range(2007, 2017):
    stats = exploratory_period_summary(
        f"{year}-01-01",
        f"{year}-06-01"
    )
    hist_ante_exploratory_rows.append(stats)

hist_ante_exploratory_df = pd.DataFrame(hist_ante_exploratory_rows)
hist_ante_exploratory = hist_ante_exploratory_df.mean(numeric_only=True)


# Exploratory candidate features
exploratory_features = {
    "hist_s1_swe_mean":
        hist_exploratory["swe_mean_kg_m2"],

    "ante_swe_mean":
        ante_exploratory["swe_mean_kg_m2"],

    "ante_swe_anom":
        ante_exploratory["swe_mean_kg_m2"]
        - hist_ante_exploratory["swe_mean_kg_m2"],

    "s1_swe_mean":
        s1_exploratory["swe_mean_kg_m2"],

    "s1_swe_anom":
        s1_exploratory["swe_mean_kg_m2"]
        - hist_exploratory["swe_mean_kg_m2"],

    "hist_s1_srad_mean":
        hist_exploratory["srad_mean_w_m2"],

    "s1_srad_mean":
        s1_exploratory["srad_mean_w_m2"],

    "s1_srad_anom":
        s1_exploratory["srad_mean_w_m2"]
        - hist_exploratory["srad_mean_w_m2"],
}


exploratory_features_df = pd.DataFrame(
    [exploratory_features],
    index=["Highline"]
).round(2)

exploratory_features_df

,hist_s1_swe_mean,ante_swe_mean,ante_swe_anom,s1_swe_mean,s1_swe_anom,hist_s1_srad_mean,s1_srad_mean,s1_srad_anom
Highline,51.74,204.92,-1.26,30.47,-21.28,453.17,443.86,-9.31


#### Step 4C observations

Exploratory snowpack and solar-radiation summaries were constructed successfully.

Historical June–September mean snow-water equivalent was approximately **51.7 kg/m²**. During the January–May 2018 antecedent period, mean SWE was approximately **204.9 kg/m²**, only about **1.3 kg/m² below** its matching 10-year historical baseline. Antecedent snowpack conditions were therefore broadly normal.

During Season 1, mean SWE was approximately **30.5 kg/m²**, about **21.3 kg/m² below** the historical June–September average. Because much of the seasonal snowpack disappears during spring and early summer, an average across the entire June–September period may not be the most informative final snow metric. Late-spring SWE or snowmelt-oriented summaries remain possible Phase-3/EDA alternatives.

Historical June–September shortwave radiation averaged approximately **453.2 W/m²**, compared with approximately **443.9 W/m² during Season 1**, an anomaly of approximately **−9.3 W/m²**. Thus, Highline's first complete post-fire growing season was only modestly below its recent historical radiation level.

SWE and shortwave radiation remain exploratory candidates rather than core V1 requirements. Their final inclusion should depend on redundancy analysis, spatial variation, and held-out predictive value.

In [11]:
daymet_candidate_features_df = pd.concat(
    [
        core_features_df,
        derived_features_df,
        exploratory_features_df,
    ],
    axis=1
)

print("Number of provisional Daymet candidate features:",
      daymet_candidate_features_df.shape[1])

daymet_candidate_features_df

Number of provisional Daymet candidate features: 28


,hist_prcp_mm,hist_tmax_c,hist_tmin_c,ante_prcp_mm,ante_tmax_c,ante_tmin_c,s1_prcp_mm,s1_tmax_c,s1_tmin_c,ante_prcp_anom_mm,...,s1_vpd_kpa,s1_vpd_anom_kpa,hist_s1_swe_mean,ante_swe_mean,ante_swe_anom,s1_swe_mean,s1_swe_anom,hist_s1_srad_mean,s1_srad_mean,s1_srad_anom
Highline,170.1,21.59,4.24,579.02,5.54,-5.44,130.35,21.08,4.77,99.772,...,0.977,-0.062,51.74,204.92,-1.26,30.47,-21.28,453.17,443.86,-9.31


#### Step 4 conclusion

A provisional Daymet candidate feature set was successfully assembled for Highline.

The current feature table contains **28 climate variables** spanning:

- recent historical climate context,
- antecedent post-fire climate,
- first-growing-season climate,
- climate anomalies relative to matching historical baselines,
- derived vapor pressure deficit,
- exploratory snowpack summaries,
- and exploratory shortwave-radiation summaries.

The 28 variables represent a **candidate pool rather than a final model specification**. Several features are mathematically or physically related, including absolute climate values, historical baselines, and their corresponding anomalies.

The final production feature set will therefore be reduced based on:

- physical interpretability,
- redundancy and collinearity,
- spatial behavior,
- data coverage,
- and held-out predictive value.

The Highline-wide row remains a proof-of-concept representation. Production Daymet features will preserve spatial variation and be associated with individual 90 m modeling patches.

### Step 5A — Validate numerical completeness and plausible ranges

Before evaluating spatial behavior, verify that the provisional Daymet candidate features are numerically well formed.

The checks in this subsection confirm that:

- no candidate feature is missing,
- no value is infinite,
- precipitation totals are non-negative,
- VPD values are non-negative,
- temperatures fall within physically plausible bounds,
- and exploratory SWE and radiation values are non-negative.

These checks are intentionally broad. Their purpose is to detect implementation or unit errors rather than define scientifically strict production thresholds.

In [12]:
import numpy as np


# Basic completeness checks
n_features = daymet_candidate_features_df.shape[1]
n_missing = int(daymet_candidate_features_df.isna().sum().sum())

numeric_values = daymet_candidate_features_df.to_numpy(dtype=float)
n_infinite = int(np.isinf(numeric_values).sum())


# Broad plausibility checks
checks = {
    "No missing values":
        n_missing == 0,

    "No infinite values":
        n_infinite == 0,

    "Absolute precipitation non-negative":
        all(
            core_features[col] >= 0
            for col in [
                "hist_prcp_mm",
                "ante_prcp_mm",
                "s1_prcp_mm",
            ]
        ),

    "Absolute VPD non-negative":
        all(
            derived_features[col] >= 0
            for col in [
                "hist_s1_vpd_kpa",
                "ante_vpd_kpa",
                "s1_vpd_kpa",
            ]
        ),

    "Temperatures within broad plausible range (-50 to 50 °C)":
        all(
            -50 <= core_features[col] <= 50
            for col in [
                "hist_tmax_c",
                "hist_tmin_c",
                "ante_tmax_c",
                "ante_tmin_c",
                "s1_tmax_c",
                "s1_tmin_c",
            ]
        ),

    "SWE values non-negative":
        all(
            exploratory_features[col] >= 0
            for col in [
                "hist_s1_swe_mean",
                "ante_swe_mean",
                "s1_swe_mean",
            ]
        ),

    "Solar radiation values non-negative":
        all(
            exploratory_features[col] >= 0
            for col in [
                "hist_s1_srad_mean",
                "s1_srad_mean",
            ]
        ),
}


print("Candidate features:", n_features)
print("Missing values:", n_missing)
print("Infinite values:", n_infinite)
print()

for check_name, passed in checks.items():
    print(f"{check_name}: {'PASS' if passed else 'FAIL'}")

Candidate features: 28
Missing values: 0
Infinite values: 0

No missing values: PASS
No infinite values: PASS
Absolute precipitation non-negative: PASS
Absolute VPD non-negative: PASS
Temperatures within broad plausible range (-50 to 50 °C): PASS
SWE values non-negative: PASS
Solar radiation values non-negative: PASS


#### Step 5A observations

All **28 provisional Daymet candidate features** passed the initial numerical validation checks.

There were:

- **0 missing values**
- **0 infinite values**

Absolute precipitation, VPD, SWE, and solar-radiation values were all non-negative, while the temperature summaries remained within broad physically plausible bounds.

These checks do not establish that every candidate feature is scientifically useful, but they indicate that the feature-construction pipeline has not produced obvious missing-data, unit-conversion, or numerical errors.

### Step 5B — Validate spatial variation across Highline

The Highline-wide feature row used during the POC is a diagnostic summary and must not obscure the underlying spatial structure of Daymet.

To verify that temporally aggregated climate features retain meaningful geographic variation, representative climate surfaces are examined across the Highline perimeter using spatial percentiles.

The selected variables represent different climate mechanisms:

- Season-1 precipitation,
- Season-1 maximum temperature,
- Season-1 vapor pressure deficit,
- antecedent snow-water equivalent.

For each variable, the 10th, 50th, and 90th spatial percentiles are calculated across Highline.

Substantial percentile differences would confirm that Daymet, despite its relatively coarse ~1 km resolution, contributes spatially varying climate information that can later be associated with individual 90 m modeling patches.

In [13]:
# Build representative spatial climate surfaces

season1_period = daymet_with_vpd.filterDate(
    "2018-06-01",
    "2018-10-01"
)

antecedent_period = daymet_with_vpd.filterDate(
    "2018-01-01",
    "2018-06-01"
)


spatial_validation_image = ee.Image.cat([
    season1_period.select("prcp")
        .sum()
        .rename("s1_prcp_mm"),

    season1_period.select("tmax")
        .mean()
        .rename("s1_tmax_c"),

    season1_period.select("vpd_kpa")
        .mean()
        .rename("s1_vpd_kpa"),

    antecedent_period.select("swe")
        .mean()
        .rename("ante_swe_mean"),
])


# Spatial percentiles across Highline
spatial_stats = spatial_validation_image.reduceRegion(
    reducer=ee.Reducer.percentile([10, 50, 90]),
    geometry=highline_geom,
    scale=1000,
    maxPixels=1e7
).getInfo()


spatial_rows = []

for band in [
    "s1_prcp_mm",
    "s1_tmax_c",
    "s1_vpd_kpa",
    "ante_swe_mean",
]:
    spatial_rows.append({
        "feature": band,
        "p10": spatial_stats[f"{band}_p10"],
        "p50": spatial_stats[f"{band}_p50"],
        "p90": spatial_stats[f"{band}_p90"],
        "p90_minus_p10":
            spatial_stats[f"{band}_p90"]
            - spatial_stats[f"{band}_p10"],
    })


spatial_validation_df = pd.DataFrame(spatial_rows).round(2)

spatial_validation_df

,feature,p10,p50,p90,p90_minus_p10
0,s1_prcp_mm,112.58,125.48,154.34,41.76
1,s1_tmax_c,19.59,21.02,22.85,3.26
2,s1_vpd_kpa,0.83,0.97,1.15,0.31
3,ante_swe_mean,124.59,199.25,285.26,160.67


#### Step 5B observations

Representative temporally aggregated Daymet features retain meaningful spatial variation across the Highline fire.

Season-1 precipitation ranged from approximately **112.6 mm at the 10th spatial percentile to 154.3 mm at the 90th percentile**, a central-80% spread of approximately **41.8 mm**.

Mean Season-1 daily maximum temperature ranged from approximately **19.6°C to 22.9°C** between the 10th and 90th percentiles, while mean VPD ranged from approximately **0.83 to 1.15 kPa**.

Antecedent January–May snow-water equivalent showed particularly strong spatial variation, ranging from approximately **124.6 to 285.3 kg/m²** between the 10th and 90th percentiles.

These results confirm that Daymet should not be reduced to a single fire-wide climate value in the production dataset. Despite its relatively coarse ~1 km resolution, the climate surfaces retain meaningful geographic variation that can be associated with individual 90 m modeling patches.

### Step 5C — Visually validate spatial climate patterns

Numerical percentiles confirm that Daymet varies across Highline, but a spatial sanity check is also useful to ensure that this variation forms coherent geographic patterns rather than isolated artifacts.

Representative temporally aggregated climate surfaces are visualized for:

- Season-1 precipitation,
- Season-1 maximum temperature,
- Season-1 VPD,
- antecedent snow-water equivalent.

The goal is qualitative validation only. These maps are not intended as final project figures or as evidence of causal relationships.

In [14]:
import geemap


# Representative temporally aggregated climate surfaces
s1_prcp_image = (
    season1_period.select("prcp")
    .sum()
    .rename("s1_prcp_mm")
)

s1_tmax_image = (
    season1_period.select("tmax")
    .mean()
    .rename("s1_tmax_c")
)

s1_vpd_image = (
    season1_period.select("vpd_kpa")
    .mean()
    .rename("s1_vpd_kpa")
)

ante_swe_image = (
    antecedent_period.select("swe")
    .mean()
    .rename("ante_swe_mean")
)


# Interactive map
Map = geemap.Map()

Map.centerObject(highline_geom, 9)

Map.addLayer(
    s1_prcp_image.clip(highline_geom),
    {"min": 105, "max": 165},
    "Season 1 precipitation (mm)"
)

Map.addLayer(
    s1_tmax_image.clip(highline_geom),
    {"min": 18, "max": 24},
    "Season 1 mean Tmax (°C)",
    False
)

Map.addLayer(
    s1_vpd_image.clip(highline_geom),
    {"min": 0.75, "max": 1.2},
    "Season 1 mean VPD (kPa)",
    False
)

Map.addLayer(
    ante_swe_image.clip(highline_geom),
    {"min": 100, "max": 300},
    "Antecedent mean SWE (kg/m²)",
    False
)

highline_fc = ee.FeatureCollection([ee.Feature(highline)])

Map.addLayer(
    highline_fc.style(
        color="000000",
        fillColor="00000000",
        width=2
    ),
    {},
    "Highline perimeter"
)
Map

Map(center=[45.357252220935166, -115.16183082821901], controls=(WidgetControl(options=['position', 'transparen…

#### Step 5C observations

Visual inspection of representative Daymet climate surfaces showed coherent broad-scale geographic structure across Highline.

Season-1 precipitation exhibited smooth spatial gradients rather than isolated pixel artifacts, consistent with the substantial precipitation variation identified through spatial percentiles.

Season-1 maximum temperature showed a distinct geographic pattern broadly consistent with the previously observed 3DEP terrain structure. Warmer conditions were concentrated through portions of the lower-elevation central corridor, while cooler conditions occurred in surrounding areas.

Season-1 VPD also displayed coherent spatial variation. Areas of relatively high VPD overlapped partially with warmer portions of Highline, as expected because atmospheric evaporative demand depends partly on temperature, while differing from the temperature surface because VPD also incorporates atmospheric moisture.

Antecedent snow-water equivalent showed strong spatial structure, with greater modeled snowpack in portions of the higher/cooler landscape and substantially lower values through parts of the central corridor.

The Daymet surfaces are visibly coarser than Landsat or 3DEP, consistent with their approximately 1 km native resolution, but retain meaningful within-fire climate variation.

Combined with the numerical coverage and percentile checks, the visual results provide no evidence of major spatial misalignment, missing-data artifacts, or implausible climate behavior for the Highline POC.

### Step 6A — Define core and exploratory Daymet feature tiers

The Highline POC produced 28 provisional Daymet candidate variables, but not every derived column should automatically enter the modeling matrix.

The production feature contract therefore separates Daymet variables into:

1. **Core features** — climate information expected to enter the initial V1 modeling workflow.
2. **Exploratory features** — scientifically plausible variables retained for EDA or later ablation testing but not required in the default model.

The core representation prioritizes:

- recent historical growing-season climate,
- actual antecedent winter/spring conditions,
- antecedent anomalies,
- Season-1 anomalies relative to the historical growing-season baseline,
- and vapor pressure deficit as the primary atmospheric-dryness representation.

For Season 1, historical baseline plus anomaly is preferred over simultaneously retaining historical baseline, absolute value, and anomaly because those three quantities are mathematically redundant.

Snowpack and solar-radiation variables remain exploratory. Daylight duration and raw vapor pressure are not included in the default feature contract.

In [15]:
# Core V1 Daymet candidate features

daymet_core_features = [
    # Historical Jun-Sep climate context
    "hist_prcp_mm",
    "hist_tmax_c",
    "hist_tmin_c",
    "hist_s1_vpd_kpa",

    # Actual antecedent Jan-May climate
    "ante_prcp_mm",
    "ante_tmax_c",
    "ante_tmin_c",
    "ante_vpd_kpa",

    # Antecedent climate anomalies
    "ante_prcp_anom_mm",
    "ante_tmax_anom_c",
    "ante_tmin_anom_c",
    "ante_vpd_anom_kpa",

    # Season-1 anomalies relative to historical Jun-Sep climate
    "s1_prcp_anom_mm",
    "s1_tmax_anom_c",
    "s1_tmin_anom_c",
    "s1_vpd_anom_kpa",
]


# Exploratory Daymet features
daymet_exploratory_features = [
    "ante_swe_mean",
    "ante_swe_anom",

    "s1_swe_mean",
    "s1_swe_anom",

    "hist_s1_srad_mean",
    "s1_srad_anom",
]


daymet_core_df = daymet_candidate_features_df[
    daymet_core_features
]

daymet_exploratory_df = daymet_candidate_features_df[
    daymet_exploratory_features
]


print("Core Daymet features:", len(daymet_core_features))
print("Exploratory Daymet features:", len(daymet_exploratory_features))

print()
print("Core feature names:")
for feature in daymet_core_features:
    print(" -", feature)

Core Daymet features: 16
Exploratory Daymet features: 6

Core feature names:
 - hist_prcp_mm
 - hist_tmax_c
 - hist_tmin_c
 - hist_s1_vpd_kpa
 - ante_prcp_mm
 - ante_tmax_c
 - ante_tmin_c
 - ante_vpd_kpa
 - ante_prcp_anom_mm
 - ante_tmax_anom_c
 - ante_tmin_anom_c
 - ante_vpd_anom_kpa
 - s1_prcp_anom_mm
 - s1_tmax_anom_c
 - s1_tmin_anom_c
 - s1_vpd_anom_kpa


#### Step 6A observations

The provisional Daymet feature contract was reduced from 28 candidate variables to **16 core features** and **6 exploratory features**.

The core feature set retains:

- historical June–September precipitation, temperature, and VPD,
- actual January–May antecedent precipitation, temperature, and VPD,
- antecedent anomalies relative to matching historical Jan–May conditions,
- and Season-1 anomalies relative to the historical June–September baseline.

This representation avoids retaining every mathematically redundant combination of historical baseline, actual value, and anomaly.

Snow-water-equivalent and solar-radiation features remain exploratory rather than default model inputs, while daylight duration and raw vapor pressure are excluded from the current V1 contract.

The feature tiers remain provisional until full-population EDA and held-out modeling evaluate redundancy and predictive value.

### Step 6B — Define model-specific Daymet access and production open items

The primary and early modeling setups intentionally receive different amounts of climate information.

The **primary model** predicts Year-5 NBR after the end of the first complete post-fire growing season. It may therefore use historical climate, post-fire antecedent climate, and Season-1 climate information through the September 30 cutoff.

The **early ablation model** is designed to represent prediction before observing first-year recovery conditions. Its Daymet feature set is therefore restricted to pre-fire historical climate context.

This distinction preserves a meaningful comparison between:

- recovery prediction based only on information available around the focal disturbance, and
- prediction after observing the first complete recovery season.

Several production details remain intentionally open for Phase 3 and EDA, including the exact antecedent-period rule across fires with different event timing, the best snowpack representation, redundancy among climate features, and the implementation of spatial aggregation from the ~1 km Daymet grid onto the common 90 m modeling grid.

In [16]:
# Model-specific Daymet feature contracts

daymet_primary_features = daymet_core_features.copy()

daymet_early_features = [
    "hist_prcp_mm",
    "hist_tmax_c",
    "hist_tmin_c",
    "hist_s1_vpd_kpa",
]


print("Primary-model Daymet features:",
      len(daymet_primary_features))

print("Early-model Daymet features:",
      len(daymet_early_features))

print()
print("Early-model feature names:")
for feature in daymet_early_features:
    print(" -", feature)

Primary-model Daymet features: 16
Early-model Daymet features: 4

Early-model feature names:
 - hist_prcp_mm
 - hist_tmax_c
 - hist_tmin_c
 - hist_s1_vpd_kpa


#### Step 6B observations

The model-specific Daymet feature contracts were defined successfully.

The **primary model** retains all **16 core Daymet features**, representing historical climate context, post-fire antecedent conditions, antecedent anomalies, and Season-1 climate anomalies through the September 30 prediction cutoff.

The **early ablation model** retains only **4 pre-fire historical climate features**:

- `hist_prcp_mm`
- `hist_tmax_c`
- `hist_tmin_c`
- `hist_s1_vpd_kpa`

Post-fire antecedent and Season-1 climate information are intentionally excluded from the early model so that the comparison with the primary model remains temporally meaningful.

Several issues remain intentionally deferred to Phase 3 and EDA, including the production antecedent-period rule, possible fire-period climate features, the final snowpack representation, radiation retention, redundancy pruning, and spatial aggregation from Daymet's ~1 km grid to the common 90 m modeling grid.

### Step 7A — Save curated Daymet validation figures

Save a small set of representative Daymet climate surfaces as reproducible project artifacts.

The selected figures span the main climate mechanisms evaluated during the POC:

- Season-1 precipitation,
- Season-1 maximum temperature,
- Season-1 vapor pressure deficit,
- antecedent snow-water equivalent.

These maps are validation artifacts rather than final presentation graphics. They preserve evidence that the temporally aggregated Daymet variables exhibit coherent spatial structure across Highline while retaining the source's visibly coarse ~1 km resolution.

In [17]:
from pathlib import Path
import urllib.request


# Resolve repo root
repo_root = (
    Path.cwd()
    if (Path.cwd() / "pyproject.toml").exists()
    else Path.cwd().parent
)

# Output directory
daymet_fig_dir = repo_root / "outputs" / "figures" / "daymet"
daymet_fig_dir.mkdir(parents=True, exist_ok=True)


# Visualization settings for the 4 Daymet diagnostic figures
daymet_figures = {
    "highline_s1_precipitation.png": (
        s1_prcp_image,
        {
            "min": 105,
            "max": 165,
            "palette": [
                "8c510a",
                "d8b365",
                "f6e8c3",
                "c7eae5",
                "5ab4ac",
                "01665e",
            ],
        },
    ),
    "highline_s1_tmax.png": (
        s1_tmax_image,
        {
            "min": 18,
            "max": 24,
            "palette": [
                "313695",
                "74add1",
                "e0f3f8",
                "fee090",
                "f46d43",
                "a50026",
            ],
        },
    ),
    "highline_s1_vpd.png": (
        s1_vpd_image,
        {
            "min": 0.75,
            "max": 1.20,
            "palette": [
                "2166ac",
                "67a9cf",
                "d1e5f0",
                "fddbc7",
                "ef8a62",
                "b2182b",
            ],
        },
    ),
    "highline_antecedent_swe.png": (
        ante_swe_image,
        {
            "min": 100,
            "max": 300,
            "palette": [
                "ffffcc",
                "c7e9b4",
                "7fcdbb",
                "41b6c4",
                "2c7fb8",
                "253494",
            ],
        },
    ),
}


# Highline perimeter overlay
perimeter = ee.Image().byte().paint(
    featureCollection=highline_fc,
    color=1,
    width=2
)

In [18]:
def export_daymet_figure(filename, image, vis_params, dimensions=1200):
    visualized = image.clip(highline_geom).visualize(**vis_params)

    perimeter_rgb = perimeter.visualize(
        min=0,
        max=1,
        palette=["000000"]
    )

    figure = visualized.blend(perimeter_rgb)

    url = figure.getThumbURL({
        "region": highline_geom.bounds(),
        "dimensions": dimensions,
        "format": "png",
    })

    output_path = daymet_fig_dir / filename
    urllib.request.urlretrieve(url, output_path)

    print("Saved:", output_path.relative_to(repo_root))


for filename, (image, vis_params) in daymet_figures.items():
    try:
        export_daymet_figure(filename, image, vis_params, dimensions=1200)
    except Exception:
        print(f"Retrying {filename} at lower resolution...")
        export_daymet_figure(filename, image, vis_params, dimensions=1000)

Saved: outputs\figures\daymet\highline_s1_precipitation.png
Saved: outputs\figures\daymet\highline_s1_tmax.png
Retrying highline_s1_vpd.png at lower resolution...
Saved: outputs\figures\daymet\highline_s1_vpd.png
Saved: outputs\figures\daymet\highline_antecedent_swe.png


#### Step 7A observations

Four representative Daymet validation figures were exported successfully:

- `highline_s1_precipitation.png`
- `highline_s1_tmax.png`
- `highline_s1_vpd.png`
- `highline_antecedent_swe.png`

The saved figures preserve the broad spatial climate gradients observed during interactive validation and visibly reflect Daymet's relatively coarse ~1 km native resolution.

These figures are retained as reproducible POC validation artifacts rather than final presentation graphics.

In [19]:
expected_figures = [
    "highline_s1_precipitation.png",
    "highline_s1_tmax.png",
    "highline_s1_vpd.png",
    "highline_antecedent_swe.png",
]

print("Candidate features:", daymet_candidate_features_df.shape[1])
print("Core features:", len(daymet_core_features))
print("Exploratory features:", len(daymet_exploratory_features))
print("Primary-model features:", len(daymet_primary_features))
print("Early-model features:", len(daymet_early_features))

print()
for filename in expected_figures:
    path = daymet_fig_dir / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

Candidate features: 28
Core features: 16
Exploratory features: 6
Primary-model features: 16
Early-model features: 4

highline_s1_precipitation.png: FOUND
highline_s1_tmax.png: FOUND
highline_s1_vpd.png: FOUND
highline_antecedent_swe.png: FOUND
